In [ ]:
#Experiment_1_MobileNetv2
#loaded path through google drive
#Please note - I have changed and tested with various parameters. Also need to update to all the directories and paths for data where needed

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2

CSV_FILE = ""
df = pd.read_csv(CSV_FILE)
print(df.columns)
#df.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

ZIP_PATH = ""
EXTRACT_PATH = ""

if not os.path.exists(EXTRACT_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall("/content/")

In [ ]:

import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import (
    TimeDistributed,
    GlobalAveragePooling2D,
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    Input
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
FRAMES_DIR = ""   
IMG_SIZE = 224
NUM_FRAMES = 16
BATCH_SIZE = 8 # we can try differnt parameters
EPOCHS = 50

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["engagement_label"])

num_classes = len(label_encoder.classes_)

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [ ]:
class VideoSequenceGenerator(Sequence):
    def __init__(self, dataframe, batch_size):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))


    def __getitem__(self, idx):
        batch_df = self.df.iloc[idx * self.batch_size:(idx + 1) * self.batch_size]
        X, y = [], []

        for _, row in batch_df.iterrows():
            video_folder = os.path.join(FRAMES_DIR, row["video_filename"])
            all_frames = sorted(os.listdir(video_folder))

            total_available = len(all_frames)
            if total_available >= NUM_FRAMES:
                indices = np.linspace(0, total_available - 1, NUM_FRAMES).astype(int)
                selected_frames = [all_frames[i] for i in indices]
            else:
                selected_frames = all_frames

            video_frames = []
            for frame_name in selected_frames:
                frame_path = os.path.join(video_folder, frame_name)
                img = cv2.imread(frame_path)
                if img is None: continue
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img / 255.0
                video_frames.append(img)

            while len(video_frames) < NUM_FRAMES:
                video_frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3)))

            X.append(video_frames)
            y.append(row["label"])

        return np.array(X), tf.keras.utils.to_categorical(y, num_classes=3)

In [ ]:
train_gen = VideoSequenceGenerator(train_df, BATCH_SIZE)
val_gen = VideoSequenceGenerator(val_df, BATCH_SIZE)

In [ ]:
from tensorflow.keras.layers import Input, TimeDistributed, GlobalAveragePooling2D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers 

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
for layer in base_model.layers:
    layer.trainable = False

video_input = Input(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3))

x = TimeDistributed(base_model)(video_input)
x = TimeDistributed(GlobalAveragePooling2D())(x)

x = Bidirectional(LSTM(128, return_sequences=False, dropout=0.3, recurrent_dropout=0.2))(x)

x = Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.01))(x)


x = Dropout(0.5)(x)

output = Dense(num_classes, activation="softmax")(x)

model = Model(video_input, output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', 
        patience=12,            
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,             
        patience=5,            
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
from sklearn.utils import class_weight

y_train = train_df["label"].values


weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

cw_dict = dict(enumerate(weights))

print(f"Mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")
print(f"Class Weights: {cw_dict}")

In [ ]:

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    class_weight=cw_dict,
    callbacks=callbacks
)

In [ ]:
preds = model.predict(val_gen, verbose=0)

y_pred = np.argmax(preds, axis=1)
y_true = val_df["label"].values

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_encoder.classes_
))

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib as plt
y_true = []
y_pred = []

for i in range(len(val_gen)):
    x, y = val_gen[i]
    preds = model.predict(x, verbose=0)
    y_true.extend(np.argmax(y, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Engagement Confusion Matrix')
plt.show()